# C-GCN Hyperparameter Tuning

## Introduction

The goal of this notebook is to showcase how to train the modified C-GCN model.

The notebook is structured as follows:
1. Setup (Dependencies, imports and function definitions)
2. Define hyperparameter grid and other static training parameters
3. Execute grid search
4. Train the best model

### ATTENTION: If using Google Colab, please upload the src_gcn.zip and data.zip files to the session storage and make sure that a CUDA-enabled runtime is being used! This notebook was tested on the T4 runtime.

## 1. Setup (Dependencies, Imports and Function Definitions)

### If using Google Colab, please upload the src_cgn.zip and data.zip files to the session storage!

In [ ]:
%pip install torchao torchtune

In [ ]:
!if [ ! -d src_cgcn ]; then unzip src_cgcn.zip; fi
!if [ ! -d data ]; then unzip data.zip; fi

In [7]:
import itertools

In [ ]:
# CD into the src folder

%cd src_cgcn
%pwd

[Errno 2] No such file or directory: '../../src_ra_cgcn'
/home/jon/uom-relation-extraction/src_ra_cgcn


'/home/jon/uom-relation-extraction/src_ra_cgcn'

In [9]:
def run_grid_search(grid, start_id=100, epochs=100, cgcn=False):
    # build default model parameter string
    default_args = f"--seed 0 --lr 0.3 --num_epoch {epochs} --mlp_layers 2 --pooling_l2 0.003 --hidden_dim 200 --optim sgd --batch_size 50 --attention True "
    if not cgcn:
        default_args += "--no-rnn "

    for i, (prune_k, pooling, posit_emb, pool_bf_attn, sent_emb, num_heads, attn_dropout) in enumerate(itertools.product(*grid)):
        # adjust model ID
        id = start_id + i
        # define new args
        new_cgcn_args = f"--id {id} --prune_k {prune_k} --pooling {pooling} "
        new_attn_args = f"--positional_emb {posit_emb} --pool_before_attention {pool_bf_attn} --use_sentence_emb {sent_emb} --num_heads {num_heads} --attention_dropout {attn_dropout} "
        # combined args
        args = default_args + new_cgcn_args + new_attn_args
        
        print(args)
        # run the bash command
        %run train.py $args

In [10]:
def train_attention_model(model_id, epochs, cgcn=False, **model_params):
    # build default model parameter string
    default_args = f"--seed 0 --lr 0.3 --num_epoch {epochs} --mlp_layers 2 --pooling_l2 0.003 --hidden_dim 200 --optim sgd --batch_size 50 --attention True "
    if not cgcn:
        default_args += "--no-rnn "

    # adjust model ID
    id = model_id
    # define new args
    new_args = f"--id {id} "
    new_args += " ".join([f"--{k} {v}" for k, v in model_params.items()])
    # combined args
    args = default_args + new_args

    print(args)
    # run the bash command
    %run train.py $args
    
def train_og_model(model_id, epochs, cgcn=False):
    # build default model parameter string
    default_args = f"--seed 0 --lr 0.3 --num_epoch {epochs} --mlp_layers 2 --pooling_l2 0.003 --hidden_dim 200 --optim sgd --batch_size 50 --attention False "
    if not cgcn:
        default_args += "--no-rnn "

    # adjust model ID
    id = model_id
    # define new args
    new_args = f"--id {id} --positional_emb none --pool_before_attention True --use_sentence_emb True --pooling max --prune_k 1 "
    # combined args
    args = default_args + new_args

    print(args)
    # run the bash command
    %run train.py $args

## 2. Define hyperparameter grid and other static training parameters

In [ ]:
# ------------ define grid search parameters ------------
# # grid
# arg_prune_k = [1, -1]
# arg_pooling = ["max"]
# arg_posit_emb = ['rot']
# arg_pool_bf_attn = [True, False]
# arg_sent_emb = [True, False]
# arg_num_heads = [1, 10]
# arg_attn_dropout = [0, 0.15]

# new grid
arg_prune_k = [1]
arg_pooling = ["max"]
arg_posit_emb = ['rot']
arg_pool_bf_attn = [True]
arg_sent_emb = [True]
arg_num_heads = [1, 4]
arg_attn_dropout = [0.1, 0.3, 0.5]

# create grid
grid = [arg_prune_k, arg_pooling, arg_posit_emb, arg_pool_bf_attn, arg_sent_emb, arg_num_heads, arg_attn_dropout]


# print size of grid
print("Grid size:", len(list(itertools.product(*grid))))

Grid size: 6


In [14]:
# --------------- set the parameters here ---------------
# set start id to not overwrite the previous runs
start_id = 300
# set epoch
epochs = 100
# set whether we run cgcn or not
cgcn = False

## 3. Execute grid search

In [15]:
# ------------------ execute grid search -----------------
run_grid_search(grid=grid, start_id=start_id, epochs=epochs, cgcn=cgcn)

1 max rot True True 1 0.1
1 max rot True True 1 0.3
1 max rot True True 1 0.5
1 max rot True True 4 0.1
1 max rot True True 4 0.3
1 max rot True True 4 0.5


## 4. Train the best model

In [11]:
train_attention_model(model_id=400, epochs=200, cgcn=False, prune_k = 1, pooling = "max", positional_emb = "rot", 
                      pool_before_attention = True, use_sentence_emb = True, num_heads = 1, attention_dropout = 0.15)

--seed 0 --lr 0.3 --num_epoch 200 --mlp_layers 2 --pooling_l2 0.003 --hidden_dim 200 --optim sgd --batch_size 50 --attention True --no-rnn --id 400 --prune_k 1 --pooling max --positional_emb rot --pool_before_attention True --use_sentence_emb True --num_heads 1 --attention_dropout 0.15
Vocab size 50115 loaded from file
Loading data from ../data/retacred with batch size 50...
1170 batches created for ../data/retacred/train.json
392 batches created for ../data/retacred/dev.json
269 batches created for ../data/retacred/test.json
Directory ./saved_models/400 do not exist; creating...
Config saved to file ./saved_models/400/config.json

Running with the following configs:
	data_dir : ../data/retacred
	vocab_dir : dataset/vocab
	emb_dim : 300
	ner_dim : 30
	pos_dim : 30
	hidden_dim : 200
	num_layers : 2
	input_dropout : 0.5
	gcn_dropout : 0.5
	word_dropout : 0.04
	topn : 10000000000.0
	lower : False
	prune_k : 1
	conv_l2 : 0
	pooling : max
	pooling_l2 : 0.003
	mlp_layers : 2
	no_adj : False


In [12]:
# train a baseline GCN model
train_og_model(model_id=500, epochs=200, cgcn=False)

--seed 0 --lr 0.3 --num_epoch 200 --mlp_layers 2 --pooling_l2 0.003 --hidden_dim 200 --optim sgd --batch_size 50 --attention False --no-rnn --id 500 --positional_emb none --pool_before_attention True --use_sentence_emb True --pooling max --prune_k 1 
Vocab size 50115 loaded from file
Loading data from ../data/retacred with batch size 50...
1170 batches created for ../data/retacred/train.json
392 batches created for ../data/retacred/dev.json
269 batches created for ../data/retacred/test.json
Directory ./saved_models/500 do not exist; creating...
Config saved to file ./saved_models/500/config.json

Running with the following configs:
	data_dir : ../data/retacred
	vocab_dir : dataset/vocab
	emb_dim : 300
	ner_dim : 30
	pos_dim : 30
	hidden_dim : 200
	num_layers : 2
	input_dropout : 0.5
	gcn_dropout : 0.5
	word_dropout : 0.04
	topn : 10000000000.0
	lower : False
	prune_k : 1
	conv_l2 : 0
	pooling : max
	pooling_l2 : 0.003
	mlp_layers : 2
	no_adj : False
	rnn : False
	rnn_hidden : 200
	rnn_